In [ ]:
import idx2numpy
import torch
from torch import nn
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt

from classes import MNISTDataset, MNISTModel

# globals
batch_size    = 100
learning_rate = 0.003
epochs        = 10

In [ ]:
# dataset i dataloader
# MNISTDataset reads idx-ubyte files directly, normalises to [0,1]
# and adds the channel dimension: (N, 1, 28, 28)

train_dataset = MNISTDataset('train-images.idx3-ubyte', 'train-labels.idx1-ubyte')
test_dataset  = MNISTDataset('t10k-images.idx3-ubyte',  't10k-labels.idx1-ubyte')

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_dataset)} samples  |  Test: {len(test_dataset)} samples")

# Visualise a batch of 10 images
images, labels = next(iter(train_dataloader))
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(images[i, 0].numpy(), cmap='gray')
    plt.title(str(labels[i].item()), fontsize=8)
    plt.axis('off')
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

In [ ]:
# model — CNN adapted for MNIST (1x28x28 -> 10 classes)
#
#  Input  (bs, 1, 28, 28)
#  Conv2d(1, 32, 3)   -> (bs, 32, 26, 26)
#  ReLU
#  MaxPool2d(2)       -> (bs, 32, 13, 13)
#  Conv2d(32, 64, 3)  -> (bs, 64, 11, 11)
#  ReLU
#  MaxPool2d(2)       -> (bs, 64,  5,  5)
#  Flatten            -> (bs, 1600)
#  Linear(1600, 128)
#  ReLU
#  Linear(128, 10)    -> logits for 10 digit classes

model = MNISTModel()
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# train loop i test loop

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * batch_size + len(X)
            print(f"  loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size        = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred       = model(X)
            test_loss += loss_fn(pred, y).item()
            correct   += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct   /= size
    print(f"  Accuracy: {(100 * correct):>0.1f}%  Avg loss: {test_loss:>8f}\n")

In [ ]:
# optimizer i loss fn
# CrossEntropyLoss = log-softmax + NLLLoss — standard for multi-class classification
# SGD with lr=0.003 follows the notebook convention
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# main — training loop
for t in range(epochs):
    print(f"Epoch {t + 1}\n{'-' * 30}")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader,  model, loss_fn)
print("Done!")

In [ ]:
# visualise predictions — green title = correct, red = wrong
model.eval()
images, labels = next(iter(test_dataloader))
with torch.no_grad():
    preds = model(images).argmax(1)

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i, 0].numpy(), cmap='gray')
    color = 'green' if preds[i] == labels[i] else 'red'
    plt.title(f"Pred: {preds[i].item()}\nTrue: {labels[i].item()}",
              color=color, fontsize=9)
    plt.axis('off')
plt.suptitle("Sample Test Predictions")
plt.tight_layout()
plt.show()